In [ ]:

def num_inputs(w):  return sum(inputs(w))
def num_edges(w):   return sum(edges(w))
def num_outputs(w): return sum(outputs(w))
def num_biases(w):  return sum(biases(w))
def num_neurons(w): return sum(neurons(w))


In [ ]:

ord = array([4,5,3,7,1,6,2,8]) - 1      # 0-based index
w2 = full((8,8), None)
for i in range(8):
	for j in range(8):
		w2[ord[i], ord[j]] = w[i,j]


In [ ]:

w[8,8] = 1; w[9,9] = 1       # bias nodes (diagonal entry 1)
w[2,8] = w[3,8] = 0.1        # bias 1 -> nodes 3, 4
w[4,9] = w[5,9] = 0.1        # bias 2 -> nodes 5, 6


In [ ]:

w = full((9,9), None)
for i in range(1,7): w[i,i] = tanh; w[i,0] = 1
w[2,1] = w[3,2] = w[4,3] = w[5,4] = w[6,1] = 1
w[7,5] = 1          # neuron 6 -> output node 8  (y1)
w[8,6] = 1          # neuron 7 -> output node 9  (y2)

source, target = array([1.0]), array([.427,.573])
xminus, x = inject_source(source, w)
xminus, x = forprop(xminus, x, w)
y = xminus[outputs(w)]
delta = inject_target(xminus, target, w, case = 'log')
delta = backprop(xminus, x, delta, w)


In [ ]:

from numpy import *
from numpy.linalg import norm
from scipy.special import softmax as sigma

def logistic_regression(A, b, num_iter, x = None):
	L = norm(A, 2)**2
	lr = 1/L
	if x is None: x = zeros(shape(A)[1])
	for _ in range(num_iter):
		g = dot(A.T, sigma(dot(A,x)) - b)
		if allclose(g, 0): break
		x = x - lr*g
	return x


In [ ]:

w = full((8,8), None)
w[2,2] = relu; w[3,3] = id; w[4,4] = sigmoid; w[5,5] = tanh
w[2,0] = w[2,1] = w[3,0] = w[3,1] = w[4,2] = 1
w[5,2] = w[4,3] = w[5,3] = w[7,4] = w[6,5] = 1


In [ ]:

ord = array([4,5,3,7,1,6,2,8]) - 1
w2 = full((8,8), None)
for i in range(8):
	for j in range(8):
		w2[ord[i], ord[j]] = w[i,j]

W1, traj1 = sample_training(X, w,  .045, 100)
W2, traj2 = sample_training(X, w2, .045, 100)


In [ ]:

dataset = hstack([sources, targets])
Wstar, trajectory = batch_training(dataset, w, .05, 3000, random = 'yes')


In [ ]:

w = full((9,9), None)
for i in range(1,7): w[i,i] = tanh; w[i,0] = 1
w[2,1] = w[3,2] = w[4,3] = w[5,4] = w[6,1] = 1
w[7,5] = 1; w[8,6] = 1
Wstar, trajectory = batch_training(dataset, w, 0.2, 100, case = 'log')


In [ ]:

from numpy import *
from matplotlib.pyplot import *

C = linspace(-1, 1, 21)

def S(delta, iters):
	xs = concatenate([C + delta, C - delta])
	# +1 for C_+, -1 for C_-
	s = concatenate([ones(len(C)), -ones(len(C))])       
	lr = 1/(1 + mean(xs**2))      # short step
	w = zeros(2)
	for _ in range(iters):
		y = w[0]*xs + w[1]
		# d loss/dy
		g = -2*s/(1 + exp(2*s*y)) 
		w -= lr*array([mean(g*xs), mean(g)])
	y = w[0]*xs + w[1]
	return mean(log(1 + exp(-2*s*y))), w

deltas = linspace(0, 2, 81)
plot(deltas, [S(d, 30000)[0] for d in deltas], '.-')
